# FinDoc RAG: Banking & Regulatory Document Assistant
**Stack:** LangChain, FAISS, BM25, sentence-transformers (BGE embeddings + cross-encoder reranker), Groq (`openai/gpt-oss-120b`)

What this notebook does:
1. Loads RBI documents/circulars with metadata
2. Builds a hybrid retriever vector + BM25 with Reciprocal Rank Fusion, followed by cross-encoder reranking
3. Generates grounded answers with page-level citations and an explicit refusal rule.
4. Evaluates retrieval and response quality using Hit Rate@5, MRR, faithfulness, and out-of-scope refusal rate.

**Before you run:** add `GROQ_API_KEY` in Colab (key icon in the left sidebar -> Secrets), or paste it when prompted.

## 1. Install

In [1]:
!pip -q install -U langchain-core langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu pypdf sentence-transformers rank_bm25 pandas requests gradio tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 739.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

## 2. Imports and config

In [2]:
import os, re, json, time, pickle, random, getpass
from pathlib import Path

import requests
import pandas as pd
from IPython.display import display, Markdown

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

#Config(these can be tweaked)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "openai/gpt-oss-120b"
FETCH_K = 30      # candidates pulled from each retriever
TOP_K = 5         # chunks sent to the LLM
REFUSAL = "The provided documents do not contain enough information to answer this question."

/tmp/ipykernel_2688/3079255654.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. API key and LLM

In [3]:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

llm = ChatGroq(model=LLM_MODEL, temperature=0)

def safe_invoke(chain, inputs, retries=4):
    """Invoke with simple exponential backoff (handles free-tier rate limits)."""
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            wait = 2 ** attempt * 2
            print(f"LLM call failed ({type(e).__name__}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("LLM call failed after retries")

## 4. Get the documents
Using:

RBI Master Directions and circulars: https://rbi.org.in -> *Notifications*

Currently we have 7 documents in this notebook

In [4]:
from google.colab import files

DOC_DIR = Path("finance_docs")
DOC_DIR.mkdir(exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    (DOC_DIR / name).write_bytes(data)
print("Saved:", list(uploaded.keys()))

Saving RBI Commercial Banks - Asset Classification, Provisioning.pdf to RBI Commercial Banks - Asset Classification, Provisioning.pdf
Saving RBI Commercial Banks – Cybersecurity, Technology Risk.pdf to RBI Commercial Banks – Cybersecurity, Technology Risk.pdf
Saving RBI Commercial Banks – Digital Payment Security .pdf to RBI Commercial Banks – Digital Payment Security .pdf
Saving RBI Commercial Banks – Fraud Risk Management.pdf to RBI Commercial Banks – Fraud Risk Management.pdf
Saving RBI Commercial Banks - Internal Ombudsman Directions.pdf to RBI Commercial Banks - Internal Ombudsman Directions.pdf
Saving RBI Digital Payments – E-mandate Framework, 2026.pdf to RBI Digital Payments – E-mandate Framework, 2026.pdf
Saving RBI Master Directions on Authorisation to operate a Payment System .pdf to RBI Master Directions on Authorisation to operate a Payment System .pdf
Saved: ['RBI Commercial Banks - Asset Classification, Provisioning.pdf', 'RBI Commercial Banks – Cybersecurity, Technology

## 5. Load PDFs with finance-specific metadata
Mechanism implemented for future metadata overrides

In [5]:
# Manual overrides, e.g. {"rbi_kyc_master_direction_2025.pdf": {"doc_type": "circular", "year": 2025}}
METADATA_OVERRIDES = {}

def infer_doc_type(fname):
    f = fname.lower()
    if "annual" in f: return "annual_report"
    if "faq" in f: return "faq"
    if any(k in f for k in ["fee", "tariff", "charges"]): return "fee_schedule"
    if any(k in f for k in ["rbi", "direction", "master", "circular", "notification"]): return "circular"
    return "other"

def infer_year(first_page_text, fname):
    pat = r"(?<!\d)(?:19|20)\d{2}(?!\d)"
    for text in (fname, first_page_text[:3000]):
        years = [int(y) for y in re.findall(pat, text) if 1990 <= int(y) <= 2026]
        if years:
            return years[0]
    return None

docs = []
for pdf in sorted(DOC_DIR.glob("*.pdf")):
    pages = PyPDFLoader(str(pdf)).load()
    if not pages:
        continue
    meta = {
        "doc_name": pdf.name,
        "doc_type": infer_doc_type(pdf.name),
        "year": infer_year(pages[0].page_content, pdf.name),
    }
    meta.update(METADATA_OVERRIDES.get(pdf.name, {}))
    for p in pages:
        p.metadata.update(meta)
    docs.extend(pages)

assert docs, "No PDFs found. Run Option A or B above."
print(f"Loaded {len(docs)} pages from {len({d.metadata['doc_name'] for d in docs})} documents")
display(pd.DataFrame(
    list({(d.metadata["doc_name"], d.metadata["doc_type"], d.metadata["year"]) for d in docs}),
    columns=["doc_name", "doc_type", "year"]
).sort_values("doc_name"))

Loaded 227 pages from 7 documents


,doc_name,doc_type,year
5,"RBI Commercial Banks - Asset Classification, P...",circular,2026
2,RBI Commercial Banks - Internal Ombudsman Dire...,circular,2025
4,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026
3,RBI Commercial Banks – Digital Payment Securit...,circular,2026
6,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022
1,"RBI Digital Payments – E-mandate Framework, 20...",circular,2026
0,RBI Master Directions on Authorisation to oper...,circular,2026


## 6. Chunking

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i
print(f"Created {len(chunks)} chunks")

Created 570 chunks


## 7. Indexes: FAISS (semantic) + BM25 (keyword) + cross-encoder reranker

In [7]:
def tokenize(text):
    """Lowercase alphanumeric tokens, so 'Fraud,' 'fraud' and 'FRAUD' all match in BM25."""
    return re.findall(r"[a-z0-9]+", text.lower())

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)

bm25 = BM25Retriever.from_documents(chunks, k=FETCH_K, preprocess_func=tokenize)

reranker = CrossEncoder(RERANK_MODEL)
print("Indexes ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Indexes ready.


## 8. Retrieval
We are comparing 4 models in evaluation stage:
- vector: embeddings only
- bm25: keywords only
- hybrid: both lists merged with Reciprocal Rank Fusion
- hybrid + rerank: hybrid candidates re-scored by a cross-encoder

In [8]:
def rrf(rank_lists, c=60):
    """Reciprocal Rank Fusion over several ranked lists of chunks."""
    scores, by_id = {}, {}
    for lst in rank_lists:
        for rank, d in enumerate(lst):
            cid = d.metadata["chunk_id"]
            scores[cid] = scores.get(cid, 0) + 1.0 / (c + rank + 1)
            by_id[cid] = d
    return [by_id[cid] for cid in sorted(scores, key=scores.get, reverse=True)]

def retrieve(query, mode="hybrid+rerank", k=TOP_K, doc_type=None, min_year=None):
    if mode == "vector":
        cands = vectorstore.similarity_search(query, k=FETCH_K)
    elif mode == "bm25":
        cands = bm25.invoke(query)
    elif mode in ("hybrid", "hybrid+rerank"):
        cands = rrf([vectorstore.similarity_search(query, k=FETCH_K), bm25.invoke(query)])
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # Metadata filters
    def keep(d):
        if doc_type and d.metadata.get("doc_type") != doc_type: return False
        if min_year and (d.metadata.get("year") or 0) < min_year: return False
        return True
    cands = [d for d in cands if keep(d)]

    if mode == "hybrid+rerank" and cands:
        cands = cands[:FETCH_K]
        scores = reranker.predict([(query, d.page_content) for d in cands])
        cands = [d for d, _ in sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)]

    return cands[:k]

## 9. Grounded prompt and chain (LCEL)

In [9]:
rag_prompt = ChatPromptTemplate.from_template("""
You are a banking and regulatory-compliance assistant.

Answer the question using ONLY the context below, which comes from RBI circulars,
bank annual reports and similar finance documents.

Rules:
- Do not invent facts, figures, dates, limits or circular numbers.
- If the context does not contain the answer, reply exactly: "{refusal}"
- Cite every claim with [Source N] matching the context blocks.
- When a document's year matters (regulations change), mention the year.
- Be concise and structured. Use bullet points for lists of requirements.

Question:
{question}

Context:
{context}
""")

# LCEL chain: prompt -> LLM -> plain string
rag_chain = rag_prompt | llm | StrOutputParser()

def format_docs(docs):
    blocks = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        blocks.append(
            f"[Source {i}] {m.get('doc_name')} | type: {m.get('doc_type')} | "
            f"year: {m.get('year')} | page: {m.get('page', 0) + 1}\n{d.page_content}"
        )
    return "\n\n---\n\n".join(blocks)

def answer_question(question, mode="hybrid+rerank", k=TOP_K, doc_type=None, min_year=None):
    docs = retrieve(question, mode=mode, k=k, doc_type=doc_type, min_year=min_year)
    if not docs:
        return {"answer": REFUSAL, "docs": []}
    answer = safe_invoke(rag_chain, {"question": question, "context": format_docs(docs), "refusal": REFUSAL})
    return {"answer": answer, "docs": docs}

def sources_table(docs):
    return pd.DataFrame([{
        "source": f"Source {i}",
        "document": d.metadata.get("doc_name"),
        "type": d.metadata.get("doc_type"),
        "year": d.metadata.get("year"),
        "page": d.metadata.get("page", 0) + 1,
    } for i, d in enumerate(docs, 1)])

def ask(question, **kwargs):
    out = answer_question(question, **kwargs)
    display(Markdown("### Answer"))
    display(Markdown(out["answer"]))
    if out["docs"]:
        display(Markdown("### Sources"))
        display(sources_table(out["docs"]))
    return out

## 10. Testing
We are testing the bot with questions it should be able to answer, we are also testing with out of scope questions which the bot should refuse

In [10]:
_ = ask("What are the responsibilities of the Board of a bank under the fraud risk management directions?")

### Answer

**Board responsibilities under the RBI Fraud Risk Management Directions (2022)**  

- **Approve a comprehensive Fraud Risk Management Policy** that:
  * Defines the roles and responsibilities of the Board (or Local Advisory Board), Board committees and Senior Management.  
  * Covers prevention, early detection, investigation, staff accountability, monitoring, recovery, reporting, an Early‑Warning‑Signal (EWS) framework and Red‑Flagging of Accounts (RFA).  
  * Incorporates compliance with the principles of natural justice in a time‑bound manner.  [Source 1]

- **Set the scope and frequency of fraud‑review processes**:
  * Determine the coverage (e.g., fraud categories, trends, sectoral/geographic concentration, detection delays, staff‑accountability delays) and periodicity of reviews conducted by the Senior Committee on Bank Management and Fraud (SCBMF). [Source 2]  
  * Decide the monetary threshold for fraud cases that must be escalated to the SCBMF, taking into account the bank’s scale and complexity. [Source 2]

- **Oversee the Early‑Warning‑Signal and Red‑Flagging framework**:
  * Approve the overall EWS/RFA framework as part of the Board‑approved policy. [Source 3]  
  * Through the Risk Management Committee of the Board (RMCB), approve the specific EWS indicators to be monitored, prescribe a turnaround time (preferably ≤ 30 days) for examining EWS alerts, and review the status of red‑flagged accounts. [Source 3]

- **Receive and act on periodic fraud‑incident reviews**:
  * Ensure that Senior Management places regular reviews of fraud incidents before the Board or the Audit Committee of the Board (ACB) as appropriate. [Source 5]

These points capture all the Board‑level duties explicitly outlined in the RBI’s 2022 Fraud Risk Management Directions.

### Sources

,source,document,type,year,page
0,Source 1,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,6
1,Source 2,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,8
2,Source 3,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,9
3,Source 4,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,17
4,Source 5,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,8


In [11]:
_ = ask("What is the timeline for reporting a fraud to the Reserve Bank of India?")

### Answer

**Timeline for reporting a fraud to the RBI (as per the 2022 RBI Commercial Banks – Fraud Risk Management circular)**  

- **Immediate reporting of the incident**  
  - The bank must inform the Fraud Monitoring Group (FMG), Department of Supervision, RBI **as soon as the fraud is detected and no later than 7 days** from its occurrence.  
  - The report is to be sent in the prescribed “Return on Bank Robberies, Dacoities, Thefts and Burglaries (RBR)” format via e‑mail to doscofrmc@rbi.org.in. [Source 3]

- **Quarterly aggregation and submission**  
  - After the initial report, the bank must compile a **quarterly Return (RBR)** covering all such cases during that quarter.  
  - This quarterly return must be filed **within 15 days after the end of the quarter** through the RBI’s online portal. [Source 3]

These are the only timelines specified in the provided documents for reporting fraud to the Reserve Bank of India.

### Sources

,source,document,type,year,page
0,Source 1,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,18
1,Source 2,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,22
2,Source 3,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,24
3,Source 4,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,23
4,Source 5,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,1


In [12]:
_ = ask("What governance requirements apply to a bank's technology risk and cybersecurity framework?")

### Answer

**Governance requirements for a bank’s technology‑risk and cybersecurity framework (RBI Directions 2026)**  

- **Overall IT Governance Framework** – The bank must establish a robust IT‑governance framework that:  
  - Defines the governance **structure and processes** needed to meet its business/strategic objectives [Source 2].  
  - Specifies the **roles, authority and responsibilities** of the Board of Directors (or Board‑level committee) and senior management [Source 2].  
  - Provides **oversight mechanisms** to ensure accountability and mitigation of IT and cyber/information‑security risks [Source 2].  

- **Key Governance Elements to be ensured** [Source 3]:  
  - The governance and information‑security structure must foster **accountability, effectiveness, efficiency**, and be staffed with **adequately skilled resources**.  
  - Objectives must be **well‑defined** and responsibilities **unambiguous** at every organisational level.  
  - The bank must have **processes for assessing and managing IT and cybersecurity risks**.  
  - **Budgetary allocations** for IT and cybersecurity must be **commensurate with the bank’s IT maturity, digital depth, threat environment and industry standards**, and used to meet the stated objectives.  
  - **Organisational arrangements** should ensure security concerns receive appropriate attention and are **escalated promptly** to the right hierarchy levels for quick action.  

- **Governance for the Cyber Security Operations Centre (CSOC)** [Source 1]:  
  - When setting up a CSOC, the bank must adopt **appropriate governance arrangements**, including:  
    - Regular **briefings of the Board/IT‑SC and senior management** on threat intelligence.  
    - Establishment of **dashboards** for effective oversight.  
    - **Formulation and enforcement of policies** with defined metrics, reporting structures and clear specifications of what is to be reported.  
    - **Timely communication** with all relevant stakeholders.  

**In summary**, the RBI’s 2026 directions require banks to embed governance at three levels: (1) a comprehensive IT‑governance framework with clear structure, roles and oversight; (2) specific governance attributes—accountability, skilled resources, risk‑assessment processes, adequate budgeting and escalation mechanisms; and (3) dedicated governance for the CSOC covering board briefings, dashboards, policy‑driven metrics and stakeholder communication.

### Sources

,source,document,type,year,page
0,Source 1,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,50
1,Source 2,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,12
2,Source 3,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,14
3,Source 4,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,22
4,Source 5,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,21


In [13]:
_ = ask("When must a bank classify a loan account as a non-performing asset?")

### Answer

**When a bank must classify a loan account as a Non‑Performing Asset (NPA)**  

*According to the RBI “Commercial Banks – Asset Classification, Provisioning and Income Recognition” Directions (2026):*

- **Triggering conditions** – a financial asset is to be classified as NPA as soon as **any one** of the following is met【Source 1†L8-L13】  
  1. **Interest and/or principal overdue for more than 90 days** on a term loan, purchased/discounted bill, etc.  
  2. **Overdraft / Cash Credit (OD/CC) classified as “out of order.”**  
  3. **Continuous 90‑day period of permitted drawings** on an OD/CC where the drawing power was sanctioned on stock/receivable statements that are older than three months.  
  4. **Crop‑loan specific rule:**  
     - For short‑duration crops – overdue for **two consecutive crop seasons** (e.g., rabi‑rabi‑rabi).  
     - For long‑duration crops – overdue for **one crop season**.  

- **Timing of classification** – the account is deemed NPA on the **day‑end processing date** when the above condition is identified. The “date of NPA” is the calendar date of the day‑end run that captures the overdue status【Source 2†L1-L7】.  

Thus, a bank must classify a loan as NPA **immediately after the day‑end process records that any of the listed conditions (e.g., >90‑day overdue) has occurred**.

### Sources

,source,document,type,year,page
0,Source 1,"RBI Commercial Banks - Asset Classification, P...",circular,2026,10
1,Source 2,"RBI Commercial Banks - Asset Classification, P...",circular,2026,17
2,Source 3,"RBI Commercial Banks - Asset Classification, P...",circular,2026,16
3,Source 4,"RBI Commercial Banks - Asset Classification, P...",circular,2026,5
4,Source 5,"RBI Commercial Banks - Asset Classification, P...",circular,2026,45


In [14]:
_ = ask("What are the functions and powers of the Internal Ombudsman of a bank?")

### Answer

**Functions and powers of the Internal Ombudsman (IO) of a bank (as per RBI Circular 2025)**  

- **Scope of complaints handled**  
  - May only consider complaints that have already been examined by the bank and are either *partially resolved* or *wholly rejected* by the bank; it does **not** receive complaints directly from the public. [Source 1]

- **Decision‑making authority**  
  - The IO can *decide* on any or all such complaints.  
  - The extent of the Deputy IO’s (Dy. IO) decision‑making power is defined by a policy approved by the bank’s Customer Service Committee. [Source 1]

- **Compensation recommendation**  
  - The IO/Dy. IO may *recommend suitable compensation* to the complainant, in line with RBI’s extant compensation guidelines and the bank’s own compensation policy (if any). [Source 1]

- **Legal representation restriction**  
  - The IO/Dy. IO **shall not** represent the bank in any legal proceedings before courts, tribunals, or other authorities. [Source 1]

- **Reporting lines**  
  - **Administrative reporting:** to the bank’s *Competent Authority* as defined in the Directions. [Source 2]  
  - **Functional reporting:** to the *Customer Service Committee* of the Board. [Source 2]

- **Support and infrastructure**  
  - The bank must provide the IO with the necessary officers, staff, office space, IT support, and other infrastructure to enable effective discharge of duties. [Source 2]

These points together define the IO’s functional remit (handling escalated complaints, deciding outcomes, recommending compensation) and its powers (decision‑making authority, compensation recommendation) along with the limits on its role (no direct public complaints, no legal representation) and its reporting/administrative framework.

### Sources

,source,document,type,year,page
0,Source 1,RBI Commercial Banks - Internal Ombudsman Dire...,circular,2025,8
1,Source 2,RBI Commercial Banks - Internal Ombudsman Dire...,circular,2025,6
2,Source 3,RBI Commercial Banks - Internal Ombudsman Dire...,circular,2025,4
3,Source 4,RBI Commercial Banks - Internal Ombudsman Dire...,circular,2025,5
4,Source 5,RBI Commercial Banks - Internal Ombudsman Dire...,circular,2025,15


In [15]:
_ = ask("What are the requirements for processing recurring payments under the e-mandate framework?")

### Answer

**Requirements for processing recurring payments under the e‑mandate framework (RBI Digital Payments – E‑mandate Framework, 2026)**  

- **Scope of applicability** – All Payment System Providers (PSPs) and Payment System Participants must follow the directions for processing recurring transactions (domestic or cross‑border) using cards, PPI or UPI. [Source 2]  
- **Customer‑facing disclosures at registration** –  
  - The e‑mandate must state the validity period and the customer’s right to withdraw it at any time; this information must be clearly communicated when the customer registers. [Source 3]  
- **Amount structure** –  
  - The mandate may be for a **fixed amount** or a **variable amount** subject to the overall cap fixed by the RBI.  
  - For variable‑amount mandates, the issuer must give the customer a facility to specify the **maximum value of any recurring transaction**. [Source 3]  
- **Pre‑transaction notification** – The customer must be provided a facility to **choose or change** the mode (e.g., SMS, email, etc.) for receiving the pre‑transaction notification from the issuer. [Source 3]  
- **Authentication (AFA) requirements** –  
  - **First transaction** under an e‑mandate must be validated through an Authentication Factor (AFA). If the first transaction is processed together with registration, AFA validation is still required. [Source 3]  
  - **Any modification or withdrawal** of an existing e‑mandate also requires AFA validation by the issuer. [Source 3]  
- **Cost to the customer** – No charges shall be levied on the customer for availing the e‑mandate facility for recurring transactions. [Source 4]  
- **Grievance redressal** – The e‑mandate offering must include details of the grievance‑redressal mechanism, added based on stakeholder feedback. [Source 1]  

These points constitute the core requirements that PSPs and participants must meet when processing recurring payments under the RBI’s 2026 e‑mandate framework.

### Sources

,source,document,type,year,page
0,Source 1,"RBI Digital Payments – E-mandate Framework, 20...",circular,2026,3
1,Source 2,"RBI Digital Payments – E-mandate Framework, 20...",circular,2026,1
2,Source 3,"RBI Digital Payments – E-mandate Framework, 20...",circular,2026,2
3,Source 4,"RBI Digital Payments – E-mandate Framework, 20...",circular,2026,4
4,Source 5,"RBI Digital Payments – E-mandate Framework, 20...",circular,2026,4


In [16]:
_ = ask("What conditions must an entity meet to be authorised to operate a payment system?")

### Answer

**Conditions an entity must satisfy to obtain RBI authorisation to operate a payment system (2026 Master Directions)**  

- **Application submission** – File a completed application in the prescribed form through the RBI portal. [Source 1]  
- **Capital adequacy** – Hold the minimum capital prescribed in the specific payment‑system guidelines issued by RBI (details posted on the RBI website). [Source 1]  
- **Net‑worth definition** – Net‑worth must consist of:  
  * Paid‑up equity capital  
  * Preference shares compulsorily convertible to equity  
  * Free reserves, share‑premium balance and capital reserves (surplus from asset sales)  
  * Adjusted for accumulated loss, intangible assets, deferred revenue expenditure and deferred tax assets (if any). [Source 1]  
- **Promoter eligibility** – The promoter (individual or corporate) must not fall under any disqualifying categories such as:  
  * Entities that have voluntarily surrendered a Certificate of Authorisation (CoA)  
  * Entities whose authorisation was refused by RBI  
  * Entities in the cooling‑off period after a prior sanction breach (cannot submit a new application during this period). [Source 3]  
  * Promoter‑group criteria listed as (C)(iii)‑(v) for corporate promoters and (D)(i)‑(iii) for individual promoters; a financial institution, scheduled commercial bank, foreign institutional investor or mutual fund is deemed a promoter only if its stake is strategic. [Source 4]  
- **Compliance with RBI terms** – The applicant must be able to comply with all terms and conditions that will be attached to the authorisation under the Payment and Settlement Systems (PSS) Act. [Source 4]  

*Only entities meeting all the above conditions are eligible for RBI authorisation to operate a payment system.*

### Sources

,source,document,type,year,page
0,Source 1,RBI Master Directions on Authorisation to oper...,circular,2026,4
1,Source 2,RBI Master Directions on Authorisation to oper...,circular,2026,7
2,Source 3,RBI Master Directions on Authorisation to oper...,circular,2026,10
3,Source 4,RBI Master Directions on Authorisation to oper...,circular,2026,4
4,Source 5,RBI Master Directions on Authorisation to oper...,circular,2026,8


cross-document and reasoning questions

Testing whether hybrid retrieval pulls from more than one document

In [17]:
_ = ask("How do the fraud risk management and cybersecurity directions differ in their incident reporting expectations?")

### Answer

**Fraud‑Risk‑Management Directions (2022)**  

- **Who reports & where:**  
  - The bank itself reports every fraud case directly to RBI through the **Fraud Monitoring Return (FMR)** portal [Source 2].  
- **Timing:**  
  - An individual fraud case must be reported **immediately and no later than 14 days** from the date the incident/account is classified as fraud [Source 2].  
- **Scope of reporting:**  
  - All fraud cases are reported, irrespective of amount [Source 2].  
  - Payment‑system‑related disputed or suspected fraudulent transactions are first reported to the **Central Payments Fraud Information Registry (CPFIR)** and, if later confirmed as fraud, are also reported via FMR [Source 2].  
- **Reporting format:**  
  - The bank selects one of the prescribed fraud categories (e.g., misappropriation of funds, forged instruments, etc.) when filing the FMR [Source 3].  
- **Other requirements:**  
  - No monthly certificates (Monthly CFR, Flash Report) are required [Source 2].  
  - Updates to an already‑filed FMR are submitted through the **FMR Update Application (FUA)** [Source 2].

**Cybersecurity Directions (2026)**  

- **Who reports & where:**  
  - The **Account Service Provider (ASP)** must inform the **bank** of any cyber‑incident affecting the bank [Source 5].  
- **Timing:**  
  - The directive only requires the ASP to notify the bank **“on a timely basis”**; no specific number of days is prescribed [Source 5].  
- **Scope of reporting:**  
  - Reporting is focused on cyber‑incidents that could impact the bank’s systems, with the bank then responsible for meeting broader regulatory incident‑management and business‑continuity obligations [Source 5].  
- **Reporting format & additional controls:**  
  - The ASP must maintain audit logs, implement alert mechanisms for log‑setting changes, and have documented procedures for incident management and BCP/DR [Source 5].

**Key differences in incident‑reporting expectations**

| Aspect | Fraud‑Risk‑Management (2022) | Cybersecurity (2026) |
|--------|------------------------------|----------------------|
| **Reporting recipient** | Directly to RBI (via FMR) | From ASP to the bank (bank then handles further reporting) |
| **Timeframe** | Fixed deadline – **≤ 14 days** after classification | Only “timely” notification; no explicit deadline |
| **Reporting trigger** | Every fraud case, regardless of amount; also payment‑system disputes via CPFIR | Any cyber‑incident affecting the bank’s IT environment |
| **Reporting mechanism** | Structured FMR portal with predefined fraud categories; updates via FUA | Informal/structured notification to the bank; supported by audit‑log and alert mechanisms |
| **Additional reporting artefacts** | No monthly certificates required | Requires audit logs, change‑alert mechanisms, documented incident‑management & BCP/DR procedures |

Thus, fraud‑risk directions impose a **strict, RBI‑directed, 14‑day reporting deadline** for each case, while cybersecurity directions place the onus on the ASP to **promptly inform the bank** without a prescribed number of days, emphasizing internal logging and procedural readiness rather than a fixed reporting schedule.

### Sources

,source,document,type,year,page
0,Source 1,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,3
1,Source 2,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,19
2,Source 3,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,18
3,Source 4,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,20
4,Source 5,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,38


In [18]:
_ = ask("Which directions require banks to have an independent audit or review function, and what do they require?")

### Answer

**Directions that mandate an independent audit / review function**

| Direction (year) | What the direction requires | Key requirements |
|------------------|----------------------------|------------------|
| **RBI – “Cybersecurity, Technology: Risk, Resilience and Assurance Framework” (2026) – Chapter VII (Information Systems Audit)** | Banks must maintain a **separate Information‑Systems (IS) Audit function** (or dedicated resources) that is distinct from day‑to‑day IT operations. | • The IS‑Audit function must be staffed with professionals possessing the required skills and competence. <br>• The Audit Committee of the Board (ACB) is responsible for oversight of IS‑Audit. <br>• An **IS‑Audit Policy** must be prepared, approved by the ACB and reviewed at least annually. <br>• The ACB must review critical IT/cyber‑security issues raised by the IS‑Audit and give direction to management. <br>• Use of **external resources** for IS‑Audit is permitted where internal capability is lacking.  [Source 1] |
| **RBI – “Fraud Risk Management” Directions (2022)** | For accounts flagged as **Red‑flagged (suspected fraud)**, banks must obtain an **independent audit** – either an external audit or an internal audit as per the board‑approved policy. | • The bank must have a **policy on engagement of external auditors**, covering due‑diligence, competency, track‑record, and contractual clauses on audit timelines and report submission. <br>• The audit (external or internal) must be conducted **at the behest of the lender** when an account is red‑flagged.  [Source 5] |

**Summary**

- The 2026 Cybersecurity/Technology Risk Directions require a dedicated, independent IS‑Audit function with a formal policy, Board‑level oversight, and the option to use external auditors where needed.  
- The 2022 Fraud Risk Management Directions require an independent audit (external or internal, per board policy) for any red‑flagged loan/account, together with a board‑approved framework for engaging external auditors.

### Sources

,source,document,type,year,page
0,Source 1,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,54
1,Source 2,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,29
2,Source 3,"RBI Commercial Banks - Asset Classification, P...",circular,2026,33
3,Source 4,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,42
4,Source 5,RBI Commercial Banks – Fraud Risk Management.pdf,circular,2022,12


out-of-scope questions (the bot should refuse)

In [19]:
_ = ask("Who won the 2011 Cricket World Cup?")

### Answer

The provided documents do not contain enough information to answer this question.

### Sources

,source,document,type,year,page
0,Source 1,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,1
1,Source 2,RBI Master Directions on Authorisation to oper...,circular,2026,7
2,Source 3,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,10
3,Source 4,"RBI Commercial Banks – Cybersecurity, Technolo...",circular,2026,54
4,Source 5,RBI Master Directions on Authorisation to oper...,circular,2026,3


In [20]:
# Finance-related but not in these documents, a harder refusal test
_ = ask("What is the current repo rate set by the RBI?")

### Answer

The provided documents do not contain enough information to answer this question.

### Sources

,source,document,type,year,page
0,Source 1,"RBI Commercial Banks - Asset Classification, P...",circular,2026,62
1,Source 2,RBI Master Directions on Authorisation to oper...,circular,2026,10
2,Source 3,RBI Commercial Banks – Digital Payment Securit...,circular,2026,29
3,Source 4,"RBI Commercial Banks - Asset Classification, P...",circular,2026,78
4,Source 5,RBI Master Directions on Authorisation to oper...,circular,2026,1


## 11. Evaluation
We evaluate the model using the following metrics

| Metric | Meaning|
|---|---|
| **Hit rate @k** | Did the chunk the question was written from appear in the top-k results? |
| **MRR** | How high up was it? (1.0 = always first) |
| **Faithfulness** | Does an LLM judge find every claim in the answer supported by the retrieved context? |
| **Refusal rate (out-of-scope)** | Does the bot correctly decline questions the documents can't answer? |



In [21]:
def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

qgen_prompt = ChatPromptTemplate.from_template("""
You are building a test set for a banking/finance document assistant.
Read the passage and write ONE specific question that can be answered using ONLY this passage.
The question must be self-contained: do not say "the passage", "this document" or "the text".
Prefer questions about concrete rules, limits, definitions, dates, requirements or figures.
Return JSON only, no extra text: {{"question": "...", "answer": "..."}}

Passage:
{passage}
""")
qgen_chain = qgen_prompt | llm | StrOutputParser()

N_EVAL = 20   # raise for a more reliable estimate (considering Groq rate limits)
random.seed(42)
candidates = [c for c in chunks if len(c.page_content) > 500]
sample = random.sample(candidates, min(N_EVAL, len(candidates)))

eval_set = []
for c in sample:
    out = extract_json(safe_invoke(qgen_chain, {"passage": c.page_content}))
    if out and out.get("question"):
        eval_set.append({"question": out["question"], "reference": out.get("answer", ""), "chunk_id": c.metadata["chunk_id"]})
    time.sleep(0.5)

print(f"Generated {len(eval_set)} test questions")
display(pd.DataFrame(eval_set)[["question", "reference"]].head(10))

Generated 20 test questions


,question,reference
0,According to the RBI Digital Payment Security ...,Controls to protect the confidentiality and in...
1,"According to the passage, what two present val...",The present value of the contractual cash flow...
2,What is the definition of “Cash Credit” and wh...,“Cash Credit” is a revolving working‑capital f...
3,What telephone number is listed for the Reserv...,022-6989 2022
4,"According to the passage, what types of expect...",Expected breach of contract that may lead to c...
5,What information must credit risk disclosures ...,They must provide information about a bank’s c...
6,"According to the RBI directions, what key deta...","The inventory must include model owners, devel..."
7,Which two senior officers must be included in ...,The Chief Financial Officer (CFO) and the Chie...
8,"If a direction, instruction, or guideline is r...",Yes; penalties may be imposed as if the direct...
9,When a bank uses its internal credit rating sy...,The internal policy must clearly define the nu...


### 11a. Compare retrieval strategies

In [22]:
MODES = ["vector", "bm25", "hybrid", "hybrid+rerank"]

def retrieval_metrics(mode, k=TOP_K):
    hits, rr = 0, []
    for item in eval_set:
        results = retrieve(item["question"], mode=mode, k=k)
        ids = [d.metadata["chunk_id"] for d in results]
        if item["chunk_id"] in ids:
            hits += 1
            rr.append(1.0 / (ids.index(item["chunk_id"]) + 1))
        else:
            rr.append(0.0)
    n = len(eval_set)
    return {"mode": mode, f"hit_rate@{k}": round(hits / n, 3), "MRR": round(sum(rr) / n, 3)}

retrieval_results = pd.DataFrame([retrieval_metrics(m) for m in MODES])
display(retrieval_results)
best_mode = retrieval_results.sort_values("MRR", ascending=False).iloc[0]["mode"]
print("Best mode by MRR:", best_mode)

,mode,hit_rate@5,MRR
0,vector,0.9,0.800
1,bm25,0.9,0.783
2,hybrid,1.0,0.817
3,hybrid+rerank,1.0,0.938


Best mode by MRR: hybrid+rerank


### 11b. Faithfulness and refusal behaviour


In [23]:
judge_prompt = ChatPromptTemplate.from_template("""
You are a strict fact-checker. Given CONTEXT and ANSWER, decide whether every factual claim in ANSWER
is directly supported by CONTEXT. Ignore citation formatting.
Return JSON only: {{"faithful": true or false, "reason": "one short sentence"}}

CONTEXT:
{context}

ANSWER:
{answer}
""")
judge_chain = judge_prompt | llm | StrOutputParser()

N_JUDGE = min(10, len(eval_set))
faith_rows = []
for item in eval_set[:N_JUDGE]:
    out = answer_question(item["question"], mode=best_mode)
    verdict = extract_json(safe_invoke(judge_chain, {"context": format_docs(out["docs"]), "answer": out["answer"]})) or {}
    faith_rows.append({
        "question": item["question"],
        "answered": REFUSAL not in out["answer"],
        "faithful": bool(verdict.get("faithful")),
        "reason": verdict.get("reason", ""),
    })
    time.sleep(0.5)

faith_df = pd.DataFrame(faith_rows)
display(faith_df)

# Out-of-scope questions
OUT_OF_SCOPE = [
    "Who won the 2011 Cricket World Cup?",
    "What is the recipe for butter chicken?",
    "How do I train a neural network in PyTorch?",
    "What is the capital of Australia?",
    "Who is the CEO of Tesla?",
    "What will the Nifty 50 close at next Friday?",
    "Write me a poem about the monsoon.",
    "What is the best smartphone to buy this year?",
]
oos_rows = []
for q in OUT_OF_SCOPE:
    out = answer_question(q, mode=best_mode)
    oos_rows.append({"question": q, "refused": REFUSAL in out["answer"]})
    time.sleep(0.5)
oos_df = pd.DataFrame(oos_rows)
display(oos_df)

,question,answered,faithful,reason
0,According to the RBI Digital Payment Security ...,True,True,All claims are directly quoted from Source 1
1,"According to the passage, what two present val...",True,True,All claims match the definition of credit loss...
2,What is the definition of “Cash Credit” and wh...,True,True,All statements match the definition and repaym...
3,What telephone number is listed for the Reserv...,True,True,All telephone numbers cited are present in the...
4,"According to the passage, what types of expect...",True,True,All listed items are explicitly mentioned in c...
5,What information must credit risk disclosures ...,True,True,All statements are directly backed by the cite...
6,"According to the RBI directions, what key deta...",True,True,All listed inventory elements are explicitly m...
7,Which two senior officers must be included in ...,True,True,Source 1 states the committee overseeing ECL i...
8,"If a direction, instruction, or guideline is r...",True,True,All statements about penalties surviving repea...
9,When a bank uses its internal credit rating sy...,True,True,All claims are directly stated in Source 1 of ...


,question,refused
0,Who won the 2011 Cricket World Cup?,True
1,What is the recipe for butter chicken?,True
2,How do I train a neural network in PyTorch?,True
3,What is the capital of Australia?,True
4,Who is the CEO of Tesla?,True
5,What will the Nifty 50 close at next Friday?,True
6,Write me a poem about the monsoon.,True
7,What is the best smartphone to buy this year?,True


### 11c. Summary

In [24]:
best_row = retrieval_results[retrieval_results["mode"] == best_mode].iloc[0]
baseline_row = retrieval_results[retrieval_results["mode"] == "vector"].iloc[0]
hit_col = f"hit_rate@{TOP_K}"

summary = {
    "documents": len({c.metadata["doc_name"] for c in chunks}),
    "chunks": len(chunks),
    "test_questions": len(eval_set),
    "baseline (vector) hit rate": baseline_row[hit_col],
    f"best ({best_mode}) hit rate": best_row[hit_col],
    "best MRR": best_row["MRR"],
    "faithfulness": round(faith_df["faithful"].mean(), 2),
    "out-of-scope refusal rate": round(oos_df["refused"].mean(), 2),
}
display(pd.Series(summary, name="value").to_frame())
print(f"Example resume line: 'Improved retrieval hit rate@{TOP_K} from {baseline_row[hit_col]:.0%} to "
      f"{best_row[hit_col]:.0%} with hybrid BM25 + vector search and cross-encoder reranking; "
      f"{summary['out-of-scope refusal rate']:.0%} of out-of-scope questions correctly refused.'")

,value
documents,7.000
chunks,570.000
test_questions,20.000
baseline (vector) hit rate,0.900
best (hybrid+rerank) hit rate,1.000
best MRR,0.938
faithfulness,1.000
out-of-scope refusal rate,1.000


Example resume line: 'Improved retrieval hit rate@5 from 90% to 100% with hybrid BM25 + vector search and cross-encoder reranking; 100% of out-of-scope questions correctly refused.'


## 12. Save and reload the index
To avoid re-embedding every session.

In [25]:
vectorstore.save_local("findoc_faiss_index")
with open("findoc_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print("Saved to ./findoc_faiss_index and ./findoc_chunks.pkl")

Saved to ./findoc_faiss_index and ./findoc_chunks.pkl
